# Spooky Author Identification

Comparison of a TF-IDF linear baseline, feature-based XGBoost and a PyTorch LSTM. We use a single stratified validation split and report **standard multiclass log loss and accuracy**. All preprocessing is fitted on the training partition.

**Data note:** The official Kaggle CSVs are not bundled. When they are absent, this notebook uses **fictional labeled sentences for a software smoke test**, not an authorship benchmark.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "author_id").exists():
    ROOT = ROOT.parent
if not (ROOT / "author_id").exists():
    raise RuntimeError("Open this notebook from the repository or notebooks/ directory")
sys.path.insert(0, str(ROOT))

from author_id.data import validate_frame, holdout_split, metrics_for, submission_frame
from author_id.features import style_features
from author_id.baseline import LinearAuthorClassifier
from author_id.classical import XGBAuthorClassifier

real_train = ROOT / "data/raw/train.csv"
real_test = ROOT / "data/raw/test.csv"
IS_DEMO = not real_train.exists()
train_file = ROOT / ("data/demo/train.csv" if IS_DEMO else "data/raw/train.csv")
test_file = ROOT / ("data/demo/test.csv" if IS_DEMO else "data/raw/test.csv")
print("DATA SOURCE:", "FICTIONAL DEMO - NO BENCHMARK INTERPRETATION" if IS_DEMO else "Original Kaggle CSVs")
train_data = validate_frame(pd.read_csv(train_file), labeled=True)
test_data = validate_frame(pd.read_csv(test_file), labeled=False)
print("Train rows:", len(train_data), "Unlabeled rows:", len(test_data))

## 1. Inspect the dataset

Class counts and text lengths give a quick view of the problem. The Kaggle classes are EAP, HPL and MWS.

In [ ]:
display(train_data["author"].value_counts().rename("rows").to_frame())
display(train_data["text"].str.len().describe().rename("characters").to_frame())

## 2. Feature engineering

The feature-based model combines linguistic statistics, TF-IDF reduced through TruncatedSVD, and out-of-fold Naive Bayes class probabilities. None of the vectorizers is fitted using validation or test text.

In [ ]:
descriptors = ["characters", "words", "unique_word_fraction", "mean_word_length", "punctuation_fraction", "uppercase_fraction", "stopword_fraction", "exclamations_and_questions"]
display(pd.DataFrame(style_features(train_data.text.iloc[:5]), columns=descriptors))

## 3. Split before fitting anything

Both models evaluate the **same** stratified 80/20 holdout. The code below uses reduced settings for a quick demonstration; use `python train.py` with the official dataset for a full run.

In [ ]:
train_part, valid_part = holdout_split(train_data, seed=40)
print("Training:", len(train_part), "Validation:", len(valid_part))

QUICK = True  # Change to False for a larger XGBoost feature representation
models = {
    "TF-IDF + Logistic Regression": LinearAuthorClassifier(),
    "TF-IDF/SVD + Naive Bayes + XGBoost": XGBAuthorClassifier(
        estimators=25 if QUICK else 350,
        word_components=8 if QUICK else 60,
        char_components=4 if QUICK else 30,
        inner_folds=2 if QUICK else 5,
    ),
}
scores = []
for name, model in models.items():
    model.fit(train_part.text, train_part.author)
    result = metrics_for(valid_part.author, model.predict_proba(valid_part.text))
    scores.append({"model": name, **result})
report = pd.DataFrame(scores).sort_values("log_loss")
display(report)

## 4. Optional PyTorch LSTM

The LSTM uses a vocabulary fitted **only on the training split**. It can train its embeddings from scratch, or use a separately obtained local `glove.6B.100d.txt` file. This is a PyTorch reimplementation; the historical notebook used Keras.

In [ ]:
RUN_LSTM = False
if RUN_LSTM:
    from author_id.neural import LSTMAuthorClassifier
    lstm = LSTMAuthorClassifier(epochs=2 if QUICK else 10, batch_size=64, max_length=80)
    lstm.fit(train_part.text, train_part.author)
    display(pd.DataFrame([{"model": "PyTorch LSTM", **metrics_for(valid_part.author, lstm.predict_proba(valid_part.text))}]))
else:
    print("Skipped LSTM. Set RUN_LSTM=True and install requirements-lstm.txt to run it.")

## 5. Create a submission

Refit the selected model on **all labeled rows** and predict the separate competition test set. Saved probabilities are ordered `EAP`, `HPL`, `MWS`. On fictional data the output has no competition meaning.

In [ ]:
selected_name = report.iloc[0]["model"]
model = models[selected_name]
model.fit(train_data.text, train_data.author)
probabilities = model.predict_proba(test_data.text)
submission = submission_frame(test_data.id, probabilities)
display(submission.head())
output = ROOT / "outputs/submission.csv"
output.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(output, index=False)
print("Saved:", output)
if IS_DEMO:
    print("These fictional samples cannot verify author classification accuracy.")

## 6. Interpret results responsibly

For genuine comparisons, use the original labeled data, rerun the full experiment, and document the dataset version, software versions and validation seed. **Do not present synthetic-demo scores or the unreproduced historical notebook outputs as new experimental results.**

For the full command-line workflow and model checkpoints, see the project README and `docs/technical_notes.md`.